In [11]:
# =============================================================================
#  UTILITAS BERSAMA
#  Dipakai oleh tahap training, inferensi, dan export hasil
# =============================================================================

import re
import warnings
import pandas as pd
import joblib
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

MODEL_PATH = "model_sqli_nb.pkl"

STOPWORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for",
    "of", "and", "with", "by", "as", "be", "was", "are", "were",
    "this", "that", "have", "has", "had", "do", "does", "did",
    "but", "so", "if", "then", "than", "its", "into", "from",
    "there", "their", "they", "will", "would", "could", "should"
}

SQL_KEYWORDS = {
    "select", "from", "where", "and", "or", "not", "is", "in",
    "like", "union", "insert", "update", "delete", "drop", "create",
    "table", "into", "values", "order", "by", "group", "having",
    "join", "on", "null", "true", "false", "case", "when", "then",
    "else", "end", "limit", "offset", "between", "exists", "all",
    "distinct", "count", "sum", "max", "min", "avg", "sleep",
    "benchmark", "char", "concat", "substring", "load_file",
    "outfile", "exec", "execute", "cast", "convert", "if"
}

def preprocess_text(text):
    """
    Fungsi preprocessing teks untuk deteksi SQL Injection.

    Tahapan:
    1. Lowercase
    2. Tokenisasi dengan regex (pisahkan kata & simbol SQL penting)
    3. Hapus stopword non-SQL
    4. Gabungkan kembali menjadi string bersih
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    text = re.sub(r'\d+', '0', text)

    tokens = re.findall(
        r"[a-z0-9_]+|--|/\*|\*/|'|\"|\(|\)|=|<|>|;|#|,|\*|\+|-|%",
        text
    )

    filtered = [
        token for token in tokens
        if token not in STOPWORDS or token in SQL_KEYWORDS
    ]

    return " ".join(filtered)


In [12]:
# =============================================================================
#  TAHAP 1 — MEMUAT DATASET
# =============================================================================

import time

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from rapidfuzz import fuzz

print("=" * 65)
print("  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES")
print("=" * 65)

DATASET_PATH = "rbsqli_dataset.csv"

df = pd.read_csv(DATASET_PATH)

# Baca berdasarkan indeks kolom (0 = input query, 2 = label)
df = df.iloc[:, [0, 2]]
df.columns = ["sentence", "label"]

print(f"\n[1] DATASET DIMUAT")
print(f"    Total data mentah : {len(df)} baris")
print(f"    Kolom             : {list(df.columns)}")


  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES

[1] DATASET DIMUAT
    Total data mentah : 10190450 baris
    Kolom             : ['sentence', 'label']


In [13]:
# =============================================================================
#  TAHAP 2 — PREPROCESSING OTOMATIS (DATA CLEANSING)
# =============================================================================

print("\n[2] PREPROCESSING OTOMATIS")

sebelum = len(df)

# Hapus baris dengan nilai kosong pada kolom utama
df.dropna(subset=["sentence", "label"], inplace=True)
print(f"    Hapus baris kosong       : {sebelum - len(df)} baris dihapus")

# Normalisasi label hanya sekali
df["label"] = (
    df["label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"yes": "1", "no": "0"})
)

# Pastikan hanya bernilai 0 atau 1
df = df[df["label"].str.match(r"^[01]$")]
df["label"] = df["label"].astype(int)
print(f"    Setelah filter label     : {len(df)} baris valid")

# Terapkan preprocessing ke kolom sentence
df["sentence_clean"] = df["sentence"].apply(preprocess_text)

# Hapus duplikasi berdasarkan teks yang sudah dibersihkan
sebelum_dup = len(df)
df.drop_duplicates(subset=["sentence_clean"], inplace=True)
df = df[df["sentence_clean"].str.strip() != ""]
print(f"    Duplikasi dihapus        : {sebelum_dup - len(df)} baris")

# Reset index setelah pembersihan
df.reset_index(drop=True, inplace=True)

print(f"    Total data bersih        : {len(df)} baris")
print(f"\n    Distribusi Kelas:")
distribusi = df["label"].value_counts()
print(f"      Label 0 (Normal) : {distribusi.get(0, 0)} sampel")
print(f"      Label 1 (SQLI)   : {distribusi.get(1, 0)} sampel")



[2] PREPROCESSING OTOMATIS
    Hapus baris kosong       : 0 baris dihapus
    Setelah filter label     : 10190450 baris valid
    Duplikasi dihapus        : 2382950 baris
    Total data bersih        : 7807500 baris

    Distribusi Kelas:
      Label 0 (Normal) : 6661440 sampel
      Label 1 (SQLI)   : 1146060 sampel


In [16]:
# =============================================================================
#  TAHAP 3 — RINGKASAN HASIL PREPROCESSING
# =============================================================================

print("\n[3] TEXT PREPROCESSING SELESAI")
print("    Contoh hasil preprocessing:")
for i in range(min(3, len(df))):
    print(f"\n    [{i+1}] Asli   : {df['sentence'].iloc[i][:70]}")
    print(f"         Bersih : {df['sentence_clean'].iloc[i][:70]}")
    print(f"         Label  : {'SQLI (1)' if df['label'].iloc[i] == 1 else 'Normal (0)'}")



[3] TEXT PREPROCESSING SELESAI
    Contoh hasil preprocessing:

    [1] Asli   : CALL created_at, updated_at FROM passwords WHERE product_id LIKE 1 NOT
         Bersih : call created_at , updated_at from passwords where product_id like 0 no
         Label  : Normal (0)

    [2] Asli   : DELETE price, status FROM payments WHERE price != NULL NOT price != 1
         Bersih : delete price , status from payments where price = null not price = 0
         Label  : Normal (0)

    [3] Asli   : EXEC('SELECT user_id, product_id FROM payments WHERE id = 1 AND " OR 1
         Bersih : exec ( ' select user_id , product_id from payments where id = 0 and " 
         Label  : SQLI (1)


In [ ]:
# =============================================================================
#  TAHAP tambahan — BALANCING KELAS DENGAN RANDOM UNDERSAMPLING
# =============================================================================

print("\n[tambahan] BALANCING KELAS")

jumlah_normal = (df["label"] == 0).sum()
jumlah_sqli = (df["label"] == 1).sum()

print(f"    Sebelum balancing: Normal = {jumlah_normal}, SQLI = {jumlah_sqli}")

if jumlah_normal > jumlah_sqli:
    df_normal = df[df["label"] == 0].sample(n=jumlah_sqli, random_state=42)
    df_sqli = df[df["label"] == 1]
    df_balanced = pd.concat([df_normal, df_sqli], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    df = df_balanced.copy()
    print(f"    Normal di-undersample menjadi {jumlah_sqli} baris")
    print(f"    Total data setelah balancing: {len(df_balanced)} baris")
else:
    df_balanced = df.copy()
    print("    Kelas normal tidak lebih banyak dari SQLI, balancing tidak dilakukan")

print("\n    Distribusi kelas setelah balancing:")
distribusi_balanced = df_balanced["label"].value_counts().sort_index()
print(f"      Label 0 (Normal) : {distribusi_balanced.get(0, 0)} sampel")
print(f"      Label 1 (SQLI)   : {distribusi_balanced.get(1, 0)} sampel")



[tambahan] BALANCING KELAS
    Sebelum balancing: Normal = 6661440, SQLI = 1146060
    Normal di-undersample menjadi 1146060 baris
    Total data setelah balancing: 2292120 baris

    Distribusi kelas setelah balancing:
      Label 0 (Normal) : 1146060 sampel
      Label 1 (SQLI)   : 1146060 sampel


In [ ]:
# =============================================================================
#  TAHAP 4 — SPLIT DATA DAN CEK LEAKAGE
# =============================================================================

source_df = df_balanced if "df_balanced" in globals() else df
print(f"\n[4] SPLIT DATASET (80:20)")
print(f"    Total data yang dipakai: {len(source_df)} sampel")

X = source_df["sentence_clean"]
y = source_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"    Data Training : {len(X_train)} sampel")
print(f"    Data Testing  : {len(X_test)} sampel")

print("\n    Cek Exact Overlap (Data Leakage):")
train_set = set(X_train)
test_set = set(X_test)
overlap = len(train_set.intersection(test_set))
print(f"    Irisan train ∩ test      : {overlap} (harus 0 — {'✅ Aman' if overlap == 0 else '⚠️ Ada Leakage!'})")

print("\n    Cek Near-Duplicate (threshold similarity > 0.95):")
sample_test = X_test.sample(n=min(300, len(X_test)), random_state=42).tolist()
sample_train = X_train.sample(n=min(500, len(X_train)), random_state=42).tolist()

count_similar = 0
for t in sample_test:
    for tr in sample_train:
        if fuzz.ratio(t, tr) > 95:
            count_similar += 1
            break

print(f"    Near-duplicate ditemukan : {count_similar} dari {len(sample_test)} sampel test")



[4] SPLIT DATASET (80:20)
    Data Training : 1833696 sampel
    Data Testing  : 458424 sampel

    Cek Exact Overlap (Data Leakage):
    Irisan train ∩ test      : 0 (harus 0 — ✅ Aman)

    Cek Near-Duplicate (threshold similarity > 0.95):


KeyboardInterrupt: 

In [8]:
# =============================================================================
#  TAHAP 5 — DEFINISI PIPELINE TF-IDF + MULTINOMIAL NAÏVE BAYES
# =============================================================================

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=1,
            max_features=5000,
            sublinear_tf=True
        )
    ),
    (
        "nb",
        MultinomialNB(alpha=0.1)
    )
])

print("\n[5] PIPELINE DIDEFINISIKAN")
print("    Algoritma  : Multinomial Naïve Bayes")
print("    Ekstraksi  : TF-IDF (unigram + bigram, max 5000 fitur)")
print("    Alpha (smoothing) : 0.1")



[5] PIPELINE DIDEFINISIKAN
    Algoritma  : Multinomial Naïve Bayes
    Ekstraksi  : TF-IDF (unigram + bigram, max 5000 fitur)
    Alpha (smoothing) : 0.1


In [9]:
# =============================================================================
#  TAHAP 6 — CROSS-VALIDATION STRATIFIED
# =============================================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print(f"\n[6] CROSS-VALIDATION (5-Fold Stratified)")
print(f"    F1 per Fold : {[round(s, 4) for s in scores]}")
print(f"    Rata-rata   : {scores.mean():.4f}")
print(f"    Std Dev     : {scores.std():.4f}")



[6] CROSS-VALIDATION (5-Fold Stratified)
    F1 per Fold : [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
    Rata-rata   : 1.0000
    Std Dev     : 0.0000


In [10]:
# =============================================================================
#  TAHAP 7 — TRAINING MODEL
# =============================================================================

pipeline.fit(X_train, y_train)

print(f"\n[7] MODEL BERHASIL DILATIH")
print(f"    Jumlah data training : {len(X_train)} sampel")


KeyboardInterrupt: 

In [9]:
# =============================================================================
#  TAHAP 8 — EVALUASI MODEL
# =============================================================================

y_pred = pipeline.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)
presisi = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n[8] HASIL EVALUASI MODEL")
print("    " + "─" * 40)
print(f"    Akurasi   : {akurasi * 100:.2f}%")
print(f"    Presisi   : {presisi * 100:.2f}%")
print(f"    Recall    : {recall * 100:.2f}%")
print(f"    F1-Score  : {f1 * 100:.2f}%")
print("    " + "─" * 40)

tn, fp, fn, tp = cm.ravel()
print(f"\n    CONFUSION MATRIX")
print(f"    {'':20} Prediksi Normal  Prediksi SQLI")
print(f"    {'Aktual Normal':<20} {tn:<17} {fp}")
print(f"    {'Aktual SQLI':<20} {fn:<17} {tp}")
print(f"\n      TP (Benar SQLI)    : {tp}")
print(f"      TN (Benar Normal)  : {tn}")
print(f"      FP (False Positive): {fp}")
print(f"      FN (False Negative): {fn}")

print("\n    CLASSIFICATION REPORT:")
print(classification_report(
    y_test, y_pred,
    target_names=["Normal (0)", "SQLI (1)"]
))



[8] HASIL EVALUASI MODEL
    ────────────────────────────────────────
    Akurasi   : 99.99%
    Presisi   : 100.00%
    Recall    : 99.97%
    F1-Score  : 99.98%
    ────────────────────────────────────────

    CONFUSION MATRIX
                         Prediksi Normal  Prediksi SQLI
    Aktual Normal        1332288           0
    Aktual SQLI          137               408031

      TP (Benar SQLI)    : 408031
      TN (Benar Normal)  : 1332288
      FP (False Positive): 0
      FN (False Negative): 137

    CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00   1332288
    SQLI (1)       1.00      1.00      1.00    408168

    accuracy                           1.00   1740456
   macro avg       1.00      1.00      1.00   1740456
weighted avg       1.00      1.00      1.00   1740456



In [10]:
# =============================================================================
#  TAHAP 9 — UJI MANUAL DETEKSI
# =============================================================================

def deteksi_sqli_dengan_waktu(input_teks: str) -> dict:
    """Fungsi deteksi SQL Injection dengan pengukuran waktu proses."""
    start_time = time.time()
    teks_bersih = preprocess_text(input_teks)
    prediksi = pipeline.predict([teks_bersih])[0]
    probabilitas = pipeline.predict_proba([teks_bersih])[0]
    latency_ms = (time.time() - start_time) * 1000

    return {
        "input": input_teks,
        "prediksi": "SQLI" if prediksi == 1 else "Normal",
        "prob_sqli": round(probabilitas[1] * 100, 2),
        "prob_normal": round(probabilitas[0] * 100, 2),
        "status": "🚫 DIBLOKIR" if prediksi == 1 else "✅ DIIZINKAN",
        "latency_ms": round(latency_ms, 2)
    }

sampel_uji = [
    "' OR '1'='1",
    "SELECT * FROM users WHERE id = 1",
    "UNION SELECT username, password FROM users--",
    "admin",
    "1; DROP TABLE users;--",
    "search=buku",
    "-1' UNION ALL SELECT NULL,NULL,NULL--",
    "username=malik&password=12345",
]

print("\n[9] UJI MANUAL DETEKSI")
print("    " + "─" * 60)

for sampel in sampel_uji:
    hasil = deteksi_sqli_dengan_waktu(sampel)
    print(f"\n    Input  : {hasil['input']}")
    print(f"    Status : {hasil['status']}")
    print(f"    P(SQLI)= {hasil['prob_sqli']}%  |  P(Normal)={hasil['prob_normal']}%")
    print(f"    Waktu  : {hasil['latency_ms']} ms  |  (Memenuhi target < 100ms)")

print("\n    " + "─" * 60)



[9] UJI MANUAL DETEKSI
    ────────────────────────────────────────────────────────────

    Input  : ' OR '1'='1
    Status : ✅ DIIZINKAN
    P(SQLI)= 26.66%  |  P(Normal)=73.34%
    Waktu  : 4.5 ms  |  (Memenuhi target < 100ms)

    Input  : SELECT * FROM users WHERE id = 1
    Status : 🚫 DIBLOKIR
    P(SQLI)= 85.13%  |  P(Normal)=14.87%
    Waktu  : 2.49 ms  |  (Memenuhi target < 100ms)

    Input  : UNION SELECT username, password FROM users--
    Status : 🚫 DIBLOKIR
    P(SQLI)= 100.0%  |  P(Normal)=0.0%
    Waktu  : 1.7 ms  |  (Memenuhi target < 100ms)

    Input  : admin
    Status : ✅ DIIZINKAN
    P(SQLI)= 38.6%  |  P(Normal)=61.4%
    Waktu  : 1.0 ms  |  (Memenuhi target < 100ms)

    Input  : 1; DROP TABLE users;--
    Status : 🚫 DIBLOKIR
    P(SQLI)= 100.0%  |  P(Normal)=0.0%
    Waktu  : 1.01 ms  |  (Memenuhi target < 100ms)

    Input  : search=buku
    Status : ✅ DIIZINKAN
    P(SQLI)= 23.45%  |  P(Normal)=76.55%
    Waktu  : 1.48 ms  |  (Memenuhi target < 100ms)

    I

In [15]:
# =============================================================================
#  TAHAP 10 — SIMPAN MODEL
# =============================================================================

MODEL_PATH = "model_sqli_nb.pkl"
joblib.dump(pipeline, MODEL_PATH)

print(f"\n[8] MODEL DISIMPAN")
print(f"    Path  : {MODEL_PATH}")
print(f"    Muat kembali dengan: pipeline = joblib.load('{MODEL_PATH}')")
print("\n" + "=" * 65)
print("  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK")
print("=" * 65)


[8] MODEL DISIMPAN
    Path  : model_sqli_nb.pkl
    Muat kembali dengan: pipeline = joblib.load('model_sqli_nb.pkl')

  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK


In [16]:
# =============================================================================
#  TAHAP 11 — LOAD MODEL DAN DATASET BARU
# =============================================================================

try:
    pipeline = joblib.load(MODEL_PATH)
    print(f"[OK] Model '{MODEL_PATH}' berhasil dimuat.\n")
except FileNotFoundError:
    print(f"[ERROR] File '{MODEL_PATH}' tidak ditemukan.")

NAMA_DATASET_BARU = "sqli_dataset.csv"

try:
    df_new = pd.read_csv(NAMA_DATASET_BARU)

    if len(df_new.columns) > 0:
        print(f"[OK] Dataset '{NAMA_DATASET_BARU}' dimuat ({len(df_new)} baris).")
        input_data = df_new.iloc[:, 0].astype(str)
        print(f"[INFO] Menguji menggunakan kolom pertama: '{df_new.columns[0]}'")
    else:
        print("[ERROR] Dataset kosong atau tidak memiliki kolom.")

except FileNotFoundError:
    print(f"[ERROR] File '{NAMA_DATASET_BARU}' tidak ditemukan di direktori /content/.")


[OK] Model 'model_sqli_nb.pkl' berhasil dimuat.

[ERROR] File 'sqli_dataset.csv' tidak ditemukan di direktori /content/.


In [12]:
# =============================================================================
#  TAHAP 12 — PREDIKSI DAN RINGKASAN HASIL
# =============================================================================

X_new = input_data.apply(preprocess_text)
y_pred_new = pipeline.predict(X_new)

df_new["prediksi_label"] = y_pred_new
df_new["status_keamanan"] = ["🚫 SQLI" if p == 1 else "✅ NORMAL" for p in y_pred_new]

print("\n--- RINGKASAN DETEKSI ---")
counts = df_new["status_keamanan"].value_counts()
print(f"Total SQL Injection : {counts.get('🚫 SQLI', 0)} query")
print(f"Total Normal        : {counts.get('✅ NORMAL', 0)} query")

print("\n--- SAMPEL HASIL PENGUJIAN ---")
display(df_new.head(10))

if len(df_new.columns) > 1 and "label" in df_new.columns.str.lower():
    col_label = [c for c in df_new.columns if c.lower() == "label"][0]
    print("\n--- LAPORAN EVALUASI ---")
    print(classification_report(df_new[col_label], y_pred_new))


NameError: name 'input_data' is not defined

In [13]:
# =============================================================================
#  TAHAP 13 — EXPORT DATA NORMAL KE CSV
# =============================================================================

try:
    df_normal = df_new[df_new["status_keamanan"] == "✅ NORMAL"]
    df_normal.to_csv("normal.csv", index=False)

    print(f"\n[OK] Berhasil menyimpan {len(df_normal)} data normal ke 'normal.csv'.")
    print("Silakan download file tersebut dari folder /content/ di sidebar kiri.")
except NameError:
    print("[ERROR] Jalankan tahap pengujian di atas terlebih dahulu.")
except Exception as e:
    print(f"[ERROR] Gagal menyimpan file: {e}")


[ERROR] Jalankan tahap pengujian di atas terlebih dahulu.
